# CIFAR-10 CNN using PyTorch

Reimplementation of [Shadab Hussain's Kaggle notebook](https://www.kaggle.com/code/shadabhussain/cifar-10-cnn-using-pytorch/notebook).

Architecture: 6 conv layers (3 blocks of 2 convs + maxpool), then 3 FC layers.

In [ ]:
from train import (
	load_cifar10, Cifar10CnnModel, DeviceDataLoader,
	get_default_device, to_device, evaluate, fit, accuracy
)
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
train_data, train_labels, test_data, test_labels = load_cifar10()
print(f'Training: {train_data.shape}, Test: {test_data.shape}')

In [ ]:
# Hyperparameters
batch_size = 128
num_epochs = 10
lr = 0.001
val_size = 5000

# Train/val split
full_dataset = TensorDataset(train_data, train_labels)
train_size = len(full_dataset) - val_size
torch.manual_seed(42)
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
test_ds = TensorDataset(test_data, test_labels)

# Data loaders
train_dl = DataLoader(train_ds, batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size * 2)
test_dl = DataLoader(test_ds, batch_size * 2)

# Device setup
device = get_default_device()
print(f'Using device: {device}')
train_dl = DeviceDataLoader(train_dl, device)
val_dl = DeviceDataLoader(val_dl, device)
test_dl = DeviceDataLoader(test_dl, device)

In [ ]:
model = to_device(Cifar10CnnModel(), device)
print(model)
print(f'\nInitial validation: {evaluate(model, val_dl)}')

In [ ]:
history = fit(num_epochs, lr, model, train_dl, val_dl)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot([x['val_acc'] for x in history], '-x')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Validation Accuracy')

ax2.plot([x['train_loss'] for x in history], '-bx', label='Training')
ax2.plot([x['val_loss'] for x in history], '-rx', label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Loss vs Epochs')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final test set evaluation
test_result = evaluate(model, test_dl)
print(f'Test accuracy: {test_result["val_acc"]:.4f}')

In [ ]:
# Predict on sample test images
import pickle
meta = pickle.load(open('datasets/cifar-10-batches-py/batches.meta', 'rb'), encoding='bytes')
label_names = [n.decode('utf-8') for n in meta[b'label_names']]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
	img = test_data[i]
	ax.imshow(img.permute(1, 2, 0))
	# Predict
	with torch.no_grad():
		out = model(to_device(img.unsqueeze(0), device))
		_, pred = torch.max(out, dim=1)
	true_label = label_names[test_labels[i].item()]
	pred_label = label_names[pred[0].item()]
	color = 'green' if true_label == pred_label else 'red'
	ax.set_title(f'{pred_label}\n({true_label})', color=color, fontsize=9)
	ax.axis('off')
plt.suptitle('Predictions (true label in parentheses)')
plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), 'cifar10-cnn.pth')
print('Model saved to cifar10-cnn.pth')